# Coffee17 DCL local-detail — fail-fast

Notebook ini hanya menjalankan DCL0 seed 123 pada validation Coffee17. Baseline EfficientNetV2-B0 GAP/HBP dipulihkan dari Hugging Face; test tidak dibuka. Checkpoint DCL disinkronkan setiap epoch agar aman saat akun/runtime Colab berganti.

In [ ]:
REPO_REF = 'agent/sni-instance-crops'
HF_REPO = 'ediprin/coffee-backbone-checkpoints'
HF_NAMESPACE = 'coffee17-dcl-v1'
DRIVE_FOLDER = 'coffee17-dcl-v1'
SEED = 123


In [ ]:
# SETUP, DATASET, DAN BASELINE — JALANKAN SEKALI
import json, os, shutil, subprocess, sys, zipfile
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
repo = Path('/content/coffee-bean-classification')
repo_url = 'https://github.com/ediprin/coffee-bean-classification.git'

def run(command, cwd=None):
    command = [str(item) for item in command]
    print('\n$', ' '.join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)

if (repo / '.git').is_dir():
    run(['git', 'fetch', 'origin', REPO_REF], cwd=repo)
    run(['git', 'checkout', REPO_REF], cwd=repo)
    run(['git', 'pull', '--ff-only', 'origin', REPO_REF], cwd=repo)
else:
    if repo.exists(): shutil.rmtree(repo)
    run(['git', 'clone', '--branch', REPO_REF, repo_url, repo])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', repo])

import torch
assert torch.cuda.is_available(), 'Aktifkan runtime GPU T4.'
token = userdata.get('HF_TOKEN')
assert token, 'Tambahkan secret HF_TOKEN dengan izin write.'
os.environ['HF_TOKEN'] = token

from huggingface_hub import HfApi, login, snapshot_download
login(token=token, add_to_git_credential=False)
print('HF USER:', HfApi().whoami(token=token)['name'])

dataset_base = Path('/content/coffee17-dcl-data')
original = dataset_base / 'original'
clean = dataset_base / 'clean'
archive = dataset_base / 'coffee17.zip'
data_root = clean / 'folds/fold_1'
suffixes = {'.jpg', '.jpeg', '.png'}

def image_count(path):
    return sum(1 for p in path.rglob('*') if p.is_file() and p.suffix.lower() in suffixes) if path.is_dir() else 0

if image_count(original / 'source') != 979:
    if original.exists(): shutil.rmtree(original)
    if archive.exists() and not zipfile.is_zipfile(archive): archive.unlink()
    run([sys.executable, '-u', '-m', 'bilinear_lmmd.data.preparation.prepare_coffee17', '--output', original, '--archive', archive, '--seed', '42'], cwd=repo)
if not (clean / 'audit.json').is_file():
    if clean.exists(): shutil.rmtree(clean)
    run([sys.executable, '-u', '-m', 'bilinear_lmmd.data.preparation.prepare_clean_grouped_folds', '--source-root', original / 'source', '--output-root', clean, '--expected-count', '979', '--folds', '5', '--seed', '42', '--validation-ratio', '0.1'], cwd=repo)
assert all((data_root / f'source/{split}').is_dir() for split in ('train', 'val', 'test'))

baseline_root = Path('/content/backbone-results')
patterns = []
for code in ('BE2G', 'BE2H'):
    patterns += [f'outputs/{code}_seed{SEED}/*', f'val_reports/{code}_seed{SEED}/*']
snapshot_download(repo_id=HF_REPO, repo_type='model', token=token, local_dir=baseline_root, allow_patterns=patterns)
for code in ('BE2G', 'BE2H'):
    checkpoint = baseline_root / f'outputs/{code}_seed{SEED}/best.pt'
    assert checkpoint.is_file(), f'Baseline tidak ditemukan di HF: {checkpoint}'

output_root = Path('/content/drive/MyDrive') / DRIVE_FOLDER
output_root.mkdir(parents=True, exist_ok=True)
print('\n=== SIAP ===')
print('GPU      :', torch.cuda.get_device_name(0))
print('DATASET  :', data_root)
print('BASELINE :', baseline_root)
print('OUTPUT   :', output_root)
print('TEST     : LOCKED')


In [ ]:
# TRAIN / RESUME DCL0 — PROGRESS TAMPIL PER BATCH
command = [
    sys.executable, '-u', '-m',
    'bilinear_lmmd.experiments.run_coffee17_dcl_screening',
    '--data-root', str(data_root),
    '--baseline-root', str(baseline_root),
    '--output-root', str(output_root),
    '--seeds', str(SEED),
    '--stage', 'dcl',
    '--evaluation-split', 'val',
    '--artifact-repo', HF_REPO,
    '--artifact-namespace', HF_NAMESPACE,
    '--artifact-sync-every', '1',
    '--artifact-required',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
run(command, cwd=repo)


In [ ]:
# HASIL VALIDATION DAN PUTUSAN FAIL-FAST
path = output_root / 'val_reports/dcl_stage_decision.json'
assert path.is_file(), f'Report belum ditemukan: {path}'
decision = json.loads(path.read_text())
print(json.dumps(decision, indent=2))
print('\nDCL0:', decision['DCL0_final']['decision'])
print('Test dibuka:', decision['test_opened'])
if decision['DCL0_final']['decision'] == 'FAIL':
    print('STOP: jangan jalankan contrastive atau seed tambahan.')
else:
    print('PASS screening: kirim hasil ini sebelum eksperimen berikutnya.')
